# Notebook 7: Build Your Own Transformer (mini-GPT)
**UCU Deep Learning Course**

In Notebook 6, we left dense networks behind for **images** and saw how convolutions, residual connections, and pretrained backbones bake the right inductive bias into vision models. In this notebook we step into the other modality that drives modern deep learning: **sequences**.

We will build a small **decoder-only transformer** (a "mini-GPT") **from scratch** — no `nn.MultiheadAttention`, no `nn.Transformer*` — and train it character by character on **Tiny Shakespeare**. By the end of the lab the model will generate text that *looks* like Shakespeare (even if it does not quite make sense), and you will be able to peek inside its attention layers to see what each head has learned.

### What you'll learn

1. Implement **scaled dot-product attention** from primitive `torch` ops.
2. Build **causal multi-head self-attention** from scratch (Q/K/V projections, head reshape, causal mask).
3. Assemble a **transformer block** (pre-LN) and stack them into a small GPT.
4. Train the model with a plain PyTorch loop and **generate text** autoregressively (temperature + top-k).
5. **Visualize per-head attention** matrices and interpret the patterns each head learns.
6. *(Optional)* Re-express attention with `torch.einsum`, and refactor the training loop into a **PyTorch Lightning** module.

### References

| Topic | Link |
|-------|------|
| Lecture 8 notes | `../../lectures/lecture 8/notes.md` |
| *Attention Is All You Need* (Vaswani et al., 2017) | https://arxiv.org/abs/1706.03762 |
| Karpathy's nanoGPT | https://github.com/karpathy/nanoGPT |
| PyTorch — building blocks | https://docs.pytorch.org/docs/stable/nn.html |
| PyTorch Lightning | https://lightning.ai/docs/pytorch/stable/ |
| Tiny Shakespeare | https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt |

> **Heads-up.** This notebook trains a model end-to-end. A GPU (CUDA) or Apple MPS device makes the lab comfortable (~3–5 min training). CPU will still work but is noticeably slower — adjust `max_iters` down if you are on CPU.


---

## Part 0 — Setup

Imports, seeds, and device selection. We follow the same `cuda → mps → cpu` pattern from Notebook 6.


In [1]:
# Colab-friendly install (skipped if already available)
try:
    import torchinfo  # noqa: F401
except ImportError:
    %pip install -q torchinfo
try:
    import tiktoken  # noqa: F401
except ImportError:
    %pip install -q tiktoken


In [2]:
from __future__ import annotations

import math
import os
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary

warnings.filterwarnings("ignore")

torch.manual_seed(42)
np.random.seed(42)

# UCU color palette (used in plots)
C1, C2, C3 = "#19326E", "#50ACB0", "#CD742A"
C4, C5, C6, C7 = "#A3477F", "#907FAB", "#4294CC", "#89A943"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    print("WARNING: training on CPU will be slow. Consider lowering max_iters below.")

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


Using device: mps
PyTorch version: 2.10.0


---

## Part 1 — From raw text to model input (~15 min)

This is the **first notebook in the series** that works with text. Before we touch the transformer, we have to answer a basic question: *what does a neural network actually see when its input is text?* Networks operate on tensors of numbers, not strings, so every text pipeline has the same two-step prelude:

```
raw text  --tokenizer-->  integer ids  --embedding-->  dense vectors  -->  transformer
```

### 1. Tokenization — strings ➜ integers

A **tokenizer** is a bookkeeping function that maps a string to a list of integer ids drawn from a fixed **vocabulary** $\mathcal{V}$. Three families are standard:

| Strategy | Vocab size | Pros | Cons |
|----------|-----------|------|------|
| **Word-level** | ~100k–1M | one id per word; meaning aligns with token boundary | enormous vocab; out-of-vocabulary (OOV) words; *walk / walks / walking* unrelated |
| **Character-level** | ~30–200 | tiny vocab; no OOV; trivial to implement | every word has to be re-assembled from characters; longer sequences |
| **Sub-word** (BPE, WordPiece, SentencePiece) | 30k–50k | compromise; frequent words intact, rare words decompose into pieces | tokenizer is itself trained on a corpus; sub-word boundaries are an artefact |

Modern industrial models (BERT, GPT-3/4, Claude, LLaMA) all use **sub-word** tokenizers, typically Byte-Pair Encoding (BPE). For this lab we will use **the GPT-2 BPE tokenizer** — the same pretrained sub-word tokenizer OpenAI shipped with GPT-2 in 2019, and the one that has been re-used (with variants) in countless downstream projects. We load it from the lightweight `tiktoken` library: it ships only the pre-computed merge rules and vocabulary (~1 MB total), no neural-network weights, no internet round-trip needed.

> **A note on data vs vocabulary.** Tiny Shakespeare contains only ~1 MB of text — ~300k BPE tokens. The GPT-2 vocabulary has ~50k entries. That ratio is far worse than what real GPT-2 was trained on (40 GB / ~8 billion tokens). Many rare tokens in the vocab will barely appear in our corpus, which limits what our mini-GPT can learn about them. Use this notebook to learn the **mechanics** of tokenization + transformer training; do not expect generations to rival ChatGPT.

### 2. Embeddings — integers ➜ dense vectors

A character id like `13` is *categorical*: ids `13` and `14` are no closer than `13` and `42`. To turn ids into something a network can compute with, the first layer of any text model is an **embedding table** — a learnable `(vocab_size, d_model)` matrix where each row is a vector for one token. Looking up the embedding for id $i$ is just **row $i$** of the table. This is what `nn.Embedding(vocab_size, d_model)` does.

The table is initialized randomly and **trained end-to-end** with everything else: gradient descent moves the row for each character so that characters appearing in similar contexts end up with similar vectors. By the end of training, the embedding space is the input geometry the transformer has learned to operate on.

### 3. Special tokens (FYI)

You will see special tokens like `[CLS]`, `[SEP]`, `[PAD]`, `[MASK]` in NLP papers. They are ordinary entries in $\mathcal{V}$ reserved for a specific role:

- `[CLS]` — prepended by **encoder** models like BERT. Its final-layer embedding becomes the input's "summary" representation used for classification. (Decoder models like GPT do not need it — they predict at *every* position.)
- `[SEP]` — separates two segments fed jointly (e.g. question + passage in QA).
- `[PAD]` — fills shorter sequences in a batch to a common length; attention is masked out at pad positions.
- `[MASK]` — the token BERT replaces ~15% of inputs with during *masked language modelling* pre-training.

The GPT-2 tokenizer we use has one special token: **`<|endoftext|>`** (id `50256`), which marks document boundaries in the training data. We will not need any of these in this notebook — Tiny Shakespeare is one continuous stream — but you will meet them constantly in real code.

### Tiny Shakespeare

Our dataset is **Tiny Shakespeare**: a single ~1 MB plain-text file with ~40k lines from Shakespeare's plays — the canonical micro-dataset for character-level language models (Karpathy's `char-rnn` / `nanoGPT`). Small enough to train on in minutes, large enough to produce convincingly Shakespearean nonsense.


In [ ]:
# Download Tiny Shakespeare (only once)
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = Path("tinyshakespeare.txt")

if not DATA_PATH.exists():
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"Saved {DATA_PATH} ({DATA_PATH.stat().st_size / 1024:.1f} KB)")

text = DATA_PATH.read_text(encoding="utf-8")
print(f"Corpus length: {len(text):,} characters")
print("First 250 chars:\n" + "-" * 40)
print(text[:250])


So far we have a plain Python string. The model cannot consume strings — only tensors of integers and floats. The next step is the **tokenizer**.

### Loading the GPT-2 tokenizer with `tiktoken`

`tiktoken.get_encoding("gpt2")` returns the pretrained GPT-2 byte-pair encoder. Two methods are all we will need:

- `tok.encode(s: str) -> list[int]` — string ➜ list of token ids.
- `tok.decode(ids: list[int]) -> str` — token ids ➜ string.

It also exposes `tok.n_vocab` for the vocabulary size.


In [ ]:
tok = tiktoken.get_encoding("gpt2")
vocab_size: int = tok.n_vocab

def encode(s: str) -> list[int]:
    return tok.encode(s)

def decode(ids: list[int]) -> str:
    return tok.decode(ids)

# --- quick round-trip check ---
sample = "Hello, world!"
assert decode(encode(sample)) == sample
print(f"vocab_size  = {vocab_size:,}")
print(f"encode('{sample}')  -> {encode(sample)}")
print(f"decode(encode('{sample}')) -> '{decode(encode(sample))}'")


### Exercise 1.1 — Inspect the tokenizer

This is mostly a **read-along** exercise — there is no implementation to fill in (the tokenizer is pretrained!). Run the cell below and look carefully at the breakdown. Notice three things:

1. The number of tokens is **far smaller** than the number of characters — BPE compresses common letter sequences into single tokens. The ratio is ~3–4× for English.
2. Tokens are **sub-words, not words**. Frequent words like `"Romeo"` may be a single token; rare or capitalized words split into pieces.
3. **Leading spaces matter** — `" the"` and `"the"` are typically different tokens. This is a GPT-2 quirk: the tokenizer encodes the space *as part of* the following word.

Try changing `sample` to your own sentence and re-run.


In [ ]:
sample = "ROMEO: O, she doth teach the torches to burn bright!"
ids = encode(sample)
print(f"original  ({len(sample)} chars):  {sample!r}")
print(f"token ids ({len(ids)} tokens):   {ids}")
print(f"compression: {len(sample) / len(ids):.2f} chars / token\n")
print("piece-by-piece breakdown:")
for i in ids:
    piece = tok.decode([i])
    print(f"  {i:>6}  {piece!r}")


> **Question 1.1** — Look at your breakdown. Find at least one example of each of the following in the same sentence:
>
> (a) A whole common word that becomes a **single token**.
>
> (b) A word that is **split** into two or more sub-word pieces.
>
> (c) A token that includes a **leading space** as part of its surface form.
>
> Why is sub-word tokenization a better trade-off than word-level (which would have ~500k vocab entries) or character-level (~100 vocab entries) for general-purpose text models?


> *(Your answer here.)*


### From ids to vectors: the embedding lookup

The tokenizer's output is integers — but the transformer expects vectors. The bridge is an **embedding layer**: a learnable `(vocab_size, d_model)` matrix where the $i$-th row is the embedding of token $i$. `nn.Embedding` is just this lookup.

Let's instantiate a tiny embedding table (`d_model = 8` so each row fits on screen) and embed the tokens of `"the the"`. The embedding values are random right now — *they would be learned during training*.


In [ ]:
# Tiny demo embedding — d_model=8 so each row fits on screen
torch.manual_seed(0)
demo_emb = nn.Embedding(vocab_size, 8)
sample_ids = torch.tensor(encode("the the"))
print(f"input string : 'the the'")
print(f"token ids    : {sample_ids.tolist()}    shape: {tuple(sample_ids.shape)}")
print("decoded pieces:", [tok.decode([i.item()]) for i in sample_ids])
print(f"\nafter nn.Embedding(vocab_size, 8):")
print(demo_emb(sample_ids))
print(f"output shape : {tuple(demo_emb(sample_ids).shape)}    (T, d_model)")


Three things to notice:

1. The output shape is `(T, d_model)` — a 2-D tensor of `T` vectors, ready to be processed by the transformer.
2. Repeated tokens (the two occurrences of `" the"`) produce **the same vector**. The embedding only knows about the token identity, not its position. That is why we will need a **positional encoding** in Part 3 — without it, the transformer would treat shuffled sequences as identical.
3. The first `"the"` (no leading space, id `the` from start-of-text) and the second `" the"` (with leading space) are **different tokens** — that is why the two rows above are not all identical.

In our full model `d_model = 128`. The embedding table has $\text{vocab\_size} \times 128 \approx 6.4$ million parameters — by far the largest single component of the model. We will share this table with the output projection later (a trick called **weight tying**) to avoid duplicating it.


### Splitting the corpus

Now encode the **entire corpus** to a single `torch.long` tensor of shape `(len(text),)`, and split it 90/10 into `train_data` and `val_data` — **contiguously**, by taking the first 90% of tokens for training and the last 10% for validation.


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data = data[n_train:]
print(f"corpus      : {len(text):,} characters  =>  {len(data):,} tokens")
print(f"compression : {len(text) / len(data):.2f} chars / token")
print(f"train tokens: {len(train_data):,}    val tokens: {len(val_data):,}")


> **Question 1.2** — Why do we split *contiguously* (first 90% / last 10%) rather than randomly shuffling token indices? What would go wrong if we shuffled?


> *(Your answer here.)*


### Exercise 1.2 — `TokenDataset`

Build a `Dataset` that yields fixed-length windows over the token stream. Each item is a pair `(x, y)` with:

- `x` of shape `(block_size,)` containing `tokens[i : i + block_size]`.
- `y` of shape `(block_size,)` containing `tokens[i + 1 : i + block_size + 1]` (next-token targets at every position).

For a window of length `block_size` we have `block_size` training signals per example (one per position) — language models are *dense* supervision.


In [ ]:
class TokenDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        # YOUR CODE HERE
        raise NotImplementedError()

    def __len__(self) -> int:
        # YOUR CODE HERE
        raise NotImplementedError()

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # YOUR CODE HERE
        raise NotImplementedError()


BLOCK_SIZE = 128
BATCH_SIZE = 64

train_dataset = TokenDataset(train_data, BLOCK_SIZE)
val_dataset = TokenDataset(val_data, BLOCK_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

# --- verification ---
xb, yb = next(iter(train_loader))
assert xb.shape == (BATCH_SIZE, BLOCK_SIZE), f"expected (64, 128), got {xb.shape}"
assert yb.shape == (BATCH_SIZE, BLOCK_SIZE)
assert (yb[:, :-1] == xb[:, 1:]).all(), "y must be x shifted by 1"
print(f"x batch shape: {xb.shape}    y batch shape: {yb.shape}")
print(f"first window:\n  x = '{decode(xb[0, :40].tolist())}'\n  y = '{decode(yb[0, :40].tolist())}'")


### What is a "language model", exactly?

The pair `(x, y)` we just built — where `y` is `x` shifted by one — *is* the autoregressive language-modelling objective:

> Given the previous tokens, predict the next one.

A window of length `block_size` gives us `block_size` separate (context, next-token) training examples in a single forward pass. Language models are densely supervised — every position contributes a gradient — and that is a big part of why they scale so well.

Let's make this concrete with the first few token positions of a real training window. Notice that with BPE the "next token" is a sub-word, not a character.


In [ ]:
xb, yb = next(iter(train_loader))
ctx = xb[0]   # one window, shape (block_size,)
tgt = yb[0]

print(f"{'pos':>3}  {'context so far':<60}  {'->':2}  {'predict':>12}")
print("-" * 86)
for t in range(8):
    context_str = decode(ctx[:t + 1].tolist()).replace("\n", "\\n")
    target_str = decode([tgt[t].item()]).replace("\n", "\\n")
    print(f"{t:>3}  {context_str!r:<60}  ->  {target_str!r:>12}")


Two ideas worth naming:

- **Teacher forcing.** During training, the input at position $n$ is always the *true* preceding context — never the model's own past predictions. This lets every position in the window contribute a gradient *simultaneously* (parallel training). At inference time we drop teacher forcing and feed back what the model itself sampled — that is what makes generation autoregressive.

- **Causal / autoregressive constraint.** Position $n$ may only depend on positions $\le n$. If the model could peek at the future, it would learn to copy `y` from `x` and the objective would be trivial. This constraint is the *only* architectural difference between **GPT-style decoders** (which we are building) and **BERT-style encoders** (which see the whole sequence bidirectionally and are trained on a different objective — masked-token prediction). Same transformer blocks, same MLPs, same LayerNorms — the decoder just adds a **causal mask** to the attention scores.

That mask is the next thing we will build.


---

## Part 2 — Self-attention from scratch (~25 min)

This is the core of the lecture. Given an input sequence of token embeddings $\mathbf{X}\in\mathbb{R}^{B\times T\times D}$, self-attention produces an output of the same shape where each token has been updated with a content-dependent mixture of the other tokens.

The recipe (from lecture §2):

1. Project the input to three matrices: **queries** $\mathbf{Q}$, **keys** $\mathbf{K}$, **values** $\mathbf{V}$, all of shape $(B, T, D)$.
2. Reshape into $H$ heads of dimension $D_h = D/H$: $(B, H, T, D_h)$.
3. Compute scaled dot-product attention per head:
$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{D_h}} + \text{mask}\right)\mathbf{V}
$$
4. Concatenate heads and project with a final output linear layer.

For a **decoder** (GPT-style), position $t$ may only attend to positions $\le t$ — we enforce this with a **causal mask** that fills the upper triangle of the attention scores with $-\infty$ before the softmax.


### Exercise 2.1 — Scaled dot-product attention

Implement the core attention function. It takes:

- `q, k, v` of shape `(B, H, T, D_h)`
- `mask` of shape `(T, T)` (or `None`) — a boolean tensor where `True` means **block this position** (i.e. set the score to $-\infty$).

Return `(out, attn)` where `out` has shape `(B, H, T, D_h)` and `attn` has shape `(B, H, T, T)` (we will use `attn` later for visualization).

Hint: `torch.matmul(q, k.transpose(-2, -1))` gives scores; `.masked_fill(mask, float('-inf'))` then `F.softmax(..., dim=-1)`.


In [ ]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    # Returns (output, attention_weights).
    # Shapes: q,k,v: (B, H, T, D_h); mask: (T, T) bool, True=block.
    # output: (B, H, T, D_h); attn: (B, H, T, T)
    # YOUR CODE HERE
    raise NotImplementedError()


# --- sanity check ---
B, H, T, Dh = 2, 4, 8, 16
q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # causal

out, attn = scaled_dot_product_attention(q, k, v, mask)
assert out.shape == (B, H, T, Dh)
assert attn.shape == (B, H, T, T)
assert torch.allclose(attn.sum(dim=-1), torch.ones(B, H, T), atol=1e-5), "rows must sum to 1"
# causal property: position 0 attends only to position 0
assert torch.allclose(attn[:, :, 0, 1:], torch.zeros(B, H, T - 1), atol=1e-7)
print("scaled_dot_product_attention OK")
print(f"output shape: {out.shape}   attn shape: {attn.shape}")


> **Question 2.1** — Why do we divide the dot products by $\sqrt{D_h}$? What happens (a) without the scaling, when $D_h$ is large, and (b) to the **gradients** that flow back through softmax in that regime?


> *(Your answer here.)*


### Exercise 2.2 — Causal multi-head self-attention

Implement `CausalSelfAttention` as a `nn.Module`. The forward pass should:

1. Take `x` of shape `(B, T, D)`.
2. Project to Q, K, V using a **single** `nn.Linear(d_model, 3 * d_model)` (more efficient than three separate linears).
3. Reshape to `(B, n_heads, T, head_dim)` where `head_dim = d_model // n_heads`.
4. Apply scaled dot-product attention with a causal mask sliced from a `(block_size, block_size)` registered buffer.
5. Reshape back to `(B, T, D)` and project through an output `nn.Linear(d_model, d_model)` + dropout.

The forward signature supports `return_attn: bool = False`. When `True`, also return the attention weights `(B, n_heads, T, T)`. We will use this in Part 5 for visualization.

Why a registered buffer? `register_buffer('tril', ...)` makes the causal mask a non-trainable tensor that moves with `model.to(device)` automatically — no manual device handling.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.1) -> None:
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_model = d_model

        # YOUR CODE HERE  ---  define qkv_proj, out_proj, attn_dropout, resid_dropout
        # and register the causal mask buffer ('tril').
        raise NotImplementedError()

    def forward(
        self, x: torch.Tensor, return_attn: bool = False
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        # YOUR CODE HERE
        # 1. Project to q, k, v
        # 2. Reshape to (B, n_heads, T, head_dim)
        # 3. Build mask from self.tril[:T, :T]
        # 4. Call scaled_dot_product_attention
        # 5. Reshape, project, dropout
        raise NotImplementedError()


# --- sanity check ---
d_model, n_heads = 128, 4
attn = CausalSelfAttention(d_model, n_heads, block_size=BLOCK_SIZE)
x = torch.randn(2, 16, d_model)
y = attn(x)
assert y.shape == x.shape
y2, a = attn(x, return_attn=True)
assert a.shape == (2, n_heads, 16, 16)
print(f"CausalSelfAttention output shape: {y.shape}")
print(f"attention weights shape: {a.shape}")
print(f"params: {sum(p.numel() for p in attn.parameters()):,}")


> **Question 2.2** — Why do we slice the mask as `tril[:T, :T]` at every forward pass instead of just using the full `(block_size, block_size)` buffer? When would this matter?


> *(Your answer here.)*


---

## Part 3 — Transformer block and mini-GPT (~20 min)

A **transformer block** wraps multi-head self-attention and a per-token MLP inside two residual connections. Modern transformers use a **pre-LN** variant where LayerNorm is applied *before* each sub-block:

$$
\mathbf{x} \leftarrow \mathbf{x} + \text{Attn}(\text{LN}(\mathbf{x})) \qquad
\mathbf{x} \leftarrow \mathbf{x} + \text{MLP}(\text{LN}(\mathbf{x}))
$$

The MLP is a 2-layer feed-forward network with hidden width $4D$ and GELU activation — this is the standard transformer FFN.


### Exercise 3.1 — `TransformerBlock`

Implement the pre-LN transformer block as described above. The MLP is:

`Linear(d_model, 4*d_model) → GELU → Linear(4*d_model, d_model) → Dropout`

The forward also takes `return_attn` and forwards it through the attention sub-module.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, block_size: int, dropout: float = 0.1) -> None:
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(
        self, x: torch.Tensor, return_attn: bool = False
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        # YOUR CODE HERE
        raise NotImplementedError()


# --- sanity check ---
block = TransformerBlock(d_model=128, n_heads=4, block_size=BLOCK_SIZE)
x = torch.randn(2, 16, 128)
y = block(x)
assert y.shape == x.shape
y2, a = block(x, return_attn=True)
assert a.shape == (2, 4, 16, 16)
print(f"TransformerBlock OK    params: {sum(p.numel() for p in block.parameters()):,}")


> **Question 3.1** — In the lecture notes the transformer is written **post-LN** (LayerNorm *after* each residual sum), but we use **pre-LN** here. What is the practical advantage of pre-LN, especially for deep stacks of blocks?


> *(Your answer here.)*


### Exercise 3.2 — `MiniGPT`

Assemble the full model. It needs:

- A token embedding `nn.Embedding(vocab_size, d_model)`.
- A **learned** positional embedding `nn.Embedding(block_size, d_model)`.
- `n_layers` of `TransformerBlock`.
- A final `LayerNorm`.
- A language-modeling head `nn.Linear(d_model, vocab_size, bias=False)`.

**Weight tying.** With `vocab_size ≈ 50k` and `d_model = 128`, the embedding table is 6.4 M parameters and the LM head would *duplicate* that. The standard trick — used in GPT-2 / GPT-3 / nanoGPT / most modern LMs — is to **share the same weights** between the input embedding and the output projection: `self.lm_head.weight = self.tok_emb.weight`. This halves the parameter count and often improves generalization (the two roles are dual: row $i$ of the embedding *is* the direction in which logit $i$ points).

Forward signature: `forward(idx: Tensor, targets: Tensor | None = None) -> (logits, loss | None)`.

- `idx` has shape `(B, T)` — token ids.
- If `targets` is provided, also compute the cross-entropy loss (flatten `(B, T, V)` logits and `(B, T)` targets to `(B*T, V)` and `(B*T,)`). Otherwise, `loss = None`.

This `(logits, loss)` convention is the nanoGPT-style API and makes the training loop a one-liner.


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        block_size: int = 128,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.block_size = block_size
        # YOUR CODE HERE  ---  define tok_emb, pos_emb, drop, blocks, ln_f, lm_head
        # Remember to apply weight tying: self.lm_head.weight = self.tok_emb.weight
        raise NotImplementedError()

    def forward(
        self, idx: torch.Tensor, targets: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        B, T = idx.shape
        assert T <= self.block_size, f"sequence length {T} > block_size {self.block_size}"
        # YOUR CODE HERE
        # 1. token + positional embeddings
        # 2. pass through blocks
        # 3. final LN + lm_head -> logits (B, T, vocab_size)
        # 4. if targets is not None, compute cross-entropy loss
        raise NotImplementedError()


# --- sanity check ---
model = MiniGPT(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=4, block_size=BLOCK_SIZE).to(device)
idx = torch.randint(0, vocab_size, (2, 16), device=device)
logits, loss = model(idx)
assert logits.shape == (2, 16, vocab_size)
assert loss is None
logits, loss = model(idx, targets=torch.randint(0, vocab_size, (2, 16), device=device))
assert loss.ndim == 0
n_params = sum(p.numel() for p in model.parameters())
n_unique = sum(p.numel() for p in model.parameters() if p.requires_grad) - (model.tok_emb.weight.numel() if model.lm_head.weight is model.tok_emb.weight else 0)
print(f"MiniGPT OK    loss = {loss.item():.4f}    expected ≈ ln(vocab_size) = {math.log(vocab_size):.4f}")
print(f"total params (with tying): {n_params:,}    (without tying would be ~{n_params + model.tok_emb.weight.numel():,})")

summary(model, input_data=idx, col_names=["input_size", "output_size", "num_params"], depth=2)


> **Question 3.2** — An untrained model should produce a cross-entropy loss close to $\ln(\text{vocab\_size})$. Why? What is this value for our vocabulary, and what does it represent?


> *(Your answer here.)*


---

## Part 4 — Train the model and generate text (~25 min)

We train with **AdamW** and a constant learning rate. Validation loss is estimated periodically on a few minibatches rather than the entire validation loader (faster, less code).

The training and evaluation helpers below are provided.


In [ ]:
@torch.no_grad()
def estimate_loss(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_batches: int = 20,
) -> dict[str, float]:
    # Estimate mean train/val loss over a handful of batches each.
    out: dict[str, float] = {}
    model.eval()
    for name, loader in [("train", train_loader), ("val", val_loader)]:
        losses = []
        for i, (xb, yb) in enumerate(loader):
            if i >= max_batches:
                break
            xb, yb = xb.to(device), yb.to(device)
            _, loss = model(xb, targets=yb)
            losses.append(loss.item())
        out[name] = float(np.mean(losses))
    model.train()
    return out


### Exercise 4.1 — Training loop

Pick reasonable hyperparameters and run the training loop. Suggested values:

- `max_iters = 3000` (drop to ~1500 on CPU)
- `eval_interval = 300`
- `learning_rate = 3e-4`

The loop should:

1. Build a `torch.optim.AdamW` optimizer over `model.parameters()` with the chosen `lr`.
2. Iterate `max_iters` times, drawing one batch per iteration from a *cycled* train iterator (so we keep training past one epoch).
3. Forward → loss → `zero_grad` → `backward` → `step`.
4. Every `eval_interval` iterations, call `estimate_loss(...)` and log to a list (`history`).

Plot train and val loss at the end.


In [ ]:
# Provided: infinite iterator over a DataLoader
def cycle(loader: DataLoader):
    while True:
        for batch in loader:
            yield batch


In [ ]:
max_iters: int = ...
eval_interval: int = ...
learning_rate: float = ...

# YOUR CODE HERE
# 1. Re-instantiate the model fresh
# 2. Define the optimizer
# 3. Training loop that yields batches from cycle(train_loader)
# 4. Every eval_interval iterations, call estimate_loss and append to history
raise NotImplementedError()

# --- plot ---
iters = [h["iter"] for h in history]
plt.figure(figsize=(10, 4))
plt.plot(iters, [h["train"] for h in history], color=C1, marker="o", label="train")
plt.plot(iters, [h["val"] for h in history], color=C3, marker="s", label="val")
plt.xlabel("iteration"); plt.ylabel("cross-entropy loss"); plt.legend()
plt.grid(True, alpha=0.3); plt.title("Training curve", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


### Exercise 4.2 — Autoregressive generation

Implement `generate`. Starting from a seed `idx` of shape `(B, T_0)`, repeatedly:

1. **Crop** `idx` to the last `block_size` tokens (the model can only attend within its context window).
2. Forward through the model to get `logits` of shape `(B, T, V)`.
3. Take the logits at the **last position only**: `logits[:, -1, :]` → shape `(B, V)`.
4. Divide by `temperature`.
5. If `top_k` is not `None`, keep only the top `k` logits — set all others to `-inf` so they have zero probability.
6. Softmax → sample with `torch.multinomial` → append to `idx`.

Repeat `max_new_tokens` times. Return the full extended `idx`.


In [ ]:
@torch.no_grad()
def generate(
    model: nn.Module,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int | None = None,
) -> torch.Tensor:
    # Autoregressive sampling. idx: (B, T_0).
    model.eval()
    # YOUR CODE HERE
    raise NotImplementedError()
    return idx


# --- generate some text ---
seed = torch.tensor([encode("\n")], dtype=torch.long, device=device)  # start with a newline
print("--- temperature=1.0, no top-k ---")
out = generate(model, seed, max_new_tokens=200, temperature=1.0)
print(decode(out[0].tolist()))


Now experiment with **temperature** and **top-k**. Generate three short samples at temperatures `0.4`, `0.8`, `1.4` with the same seed, and one with `top_k=10`.


In [ ]:
# YOUR CODE HERE — generate four samples (3 temperatures + 1 top-k) and print them
raise NotImplementedError()


> **Question 4.1** — How does temperature affect the diversity and quality of the generated text? Connect this to the softmax formula: what happens to the probability distribution as $\tau \to 0$ and $\tau \to \infty$?


> *(Your answer here.)*


---

## Part 5 — Visualizing attention (~15 min)

The transformer routes information between tokens via attention weights. Each weight $a_{n,m}^{(h, \ell)}$ tells us how much position $n$ at head $h$ in layer $\ell$ pulled information from position $m$. Visualizing these matrices is the easiest way to see what the model has learned.

We extend `MiniGPT.forward_with_attn` to return the attention matrix from every layer, then plot all `n_layers × n_heads` matrices as a grid of heatmaps.


In [ ]:
@torch.no_grad()
def forward_with_attn(model: MiniGPT, idx: torch.Tensor) -> list[torch.Tensor]:
    # Run the model and collect attention weights from every block.
    # Returns a list of length n_layers, each element a tensor of shape (B, n_heads, T, T).
    model.eval()
    B, T = idx.shape
    pos = torch.arange(T, device=idx.device)
    x = model.tok_emb(idx) + model.pos_emb(pos)
    x = model.drop(x)
    attns: list[torch.Tensor] = []
    for block in model.blocks:
        x, a = block(x, return_attn=True)
        attns.append(a)
    return attns


In [ ]:
# Pick a recognisable Shakespearean snippet
prompt = "ROMEO: O, she doth teach the torches"
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
attns = forward_with_attn(model, idx)

n_layers = len(attns)
n_heads = attns[0].shape[1]
T = idx.shape[1]
# Each tick label is the sub-word piece for that position
tokens = [tok.decode([i]).replace(" ", "·") for i in idx[0].tolist()]
print(f"layers: {n_layers}    heads: {n_heads}    sequence length: {T} sub-word tokens")
print(f"tokens: {tokens}")

fig, axes = plt.subplots(n_layers, n_heads, figsize=(3 * n_heads, 3 * n_layers))
for L in range(n_layers):
    for H in range(n_heads):
        ax = axes[L, H] if n_layers > 1 else axes[H]
        a = attns[L][0, H].cpu().numpy()
        ax.imshow(a, cmap="viridis", aspect="auto", vmin=0, vmax=a.max())
        ax.set_xticks(range(T)); ax.set_xticklabels(tokens, fontsize=7, rotation=90)
        ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=7)
        ax.set_title(f"L{L} H{H}", fontsize=9)
fig.suptitle("Attention weights (rows = query position, cols = key position)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


> **Question 5.1** — Look across the grid of attention heatmaps.
>
> (a) Find one head that mostly attends to the **immediately previous token** (a strong sub-diagonal). What might this head be doing?
>
> (b) Find one head whose attention is much more **diffuse** or **non-local**. In which layer do diffuse patterns tend to appear — early or late?
>
> (c) Why are *all* heatmaps lower-triangular?


> *(Your answer here.)*


---

## Part 6 — Optional extras

These sections introduce two industry-standard tools you will encounter in real codebases. They are **optional** and can be done in any order. Allocate ~20–30 minutes total if you tackle both.


### Optional 6.1 — Rewrite attention with `torch.einsum`

The two `torch.matmul` calls in scaled dot-product attention are clean for 4-D tensors, but they hide what is actually a sum over a single axis. `torch.einsum` makes the indices explicit and generalizes effortlessly to higher-rank tensors (cross-attention, batched relative positions, etc.).

Recall the operation:

- Scores: $\mathbf{S}_{b,h,i,j} = \sum_d \mathbf{Q}_{b,h,i,d}\,\mathbf{K}_{b,h,j,d}$
- Output: $\mathbf{O}_{b,h,i,d} = \sum_j \mathbf{A}_{b,h,i,j}\,\mathbf{V}_{b,h,j,d}$

In einsum notation:

- `scores = einsum("bhid,bhjd->bhij", q, k)`
- `out    = einsum("bhij,bhjd->bhid", attn, v)`

Implement `scaled_dot_product_attention_einsum` and assert it matches the original to within `atol=1e-6`.


In [ ]:
def scaled_dot_product_attention_einsum(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    # YOUR CODE HERE
    raise NotImplementedError()


# --- equivalence check ---
torch.manual_seed(0)
B, H, T, Dh = 2, 4, 8, 16
q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
v = torch.randn(B, H, T, Dh)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

out_ref, attn_ref = scaled_dot_product_attention(q, k, v, mask)
out_ein, attn_ein = scaled_dot_product_attention_einsum(q, k, v, mask)

assert torch.allclose(out_ref, out_ein, atol=1e-6)
assert torch.allclose(attn_ref, attn_ein, atol=1e-6)
print("einsum version matches matmul version ✓")


> **Question 6.1** — Write down the einsum string for **cross-attention**, where queries come from one sequence of length $T_q$ and keys/values from another of length $T_k$. The score and output shapes should follow naturally from the indices.


> *(Your answer here.)*


### Optional 6.2 — Refactor the trainer into PyTorch Lightning

PyTorch Lightning is a thin wrapper over PyTorch that separates *what the model does* (`training_step`) from *how it is trained* (device placement, gradient accumulation, mixed precision, distributed training, checkpointing, logging, …). The model and data are unchanged; we re-package the loop into a `LightningModule` and let `pl.Trainer` run it.

Read the [Lightning quickstart](https://lightning.ai/docs/pytorch/stable/starter/introduction.html) once before continuing. The two methods we will use are:

- `training_step(self, batch, batch_idx)` — called once per batch with autograd enabled. Return the loss; Lightning handles `zero_grad`/`backward`/`step` for you.
- `configure_optimizers(self)` — return the optimizer (and optionally schedulers).

Run with `pl.Trainer(max_epochs=1, accelerator='auto', devices=1)`. Because Tiny Shakespeare is small and we use iteration-based training, `max_epochs=1` with a limited number of batches is plenty.


In [ ]:
try:
    import pytorch_lightning as pl
except ImportError:
    %pip install -q pytorch_lightning
    import pytorch_lightning as pl


In [ ]:
class LitMiniGPT(pl.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        block_size: int = 128,
        learning_rate: float = 3e-4,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.model = MiniGPT(vocab_size, d_model, n_heads, n_layers, block_size)
        self.learning_rate = learning_rate

    def training_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        # YOUR CODE HERE — unpack batch, forward through self.model, return loss
        raise NotImplementedError()

    def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        # YOUR CODE HERE — same as training_step but log "val_loss"
        raise NotImplementedError()

    def configure_optimizers(self) -> torch.optim.Optimizer:
        # YOUR CODE HERE
        raise NotImplementedError()


lit_model = LitMiniGPT(vocab_size=vocab_size, block_size=BLOCK_SIZE)
trainer = pl.Trainer(
    max_epochs=1,
    accelerator="auto",
    devices=1,
    limit_train_batches=500,
    limit_val_batches=20,
    log_every_n_steps=50,
)
trainer.fit(lit_model, train_loader, val_loader)


> **Question 6.2** — Compare the Lightning code to the plain training loop from Exercise 4.1. What is automated by `pl.Trainer` that you had to write by hand? Name at least three concrete things.


> *(Your answer here.)*


---

## Done!

You just built and trained a transformer from scratch — the same architectural skeleton (with vastly more parameters and data) powers GPT-4, Claude, Gemini, LLaMA, and essentially every modern foundation model.

If you want to go further:

- Increase `d_model`, `n_layers`, and `block_size` and train longer — the model gets noticeably better at Shakespeare.
- Swap the learned positional embedding for a sinusoidal one (lecture §3.3) and compare.
- Replace the Tiny Shakespeare corpus with your own text (a code file, a book, song lyrics) and retrain — the architecture does not change at all.
- Re-implement *cross-attention* and build an encoder–decoder for a tiny seq2seq task (e.g. reversing strings).

See you in lecture 9 — **generative models**.
